In [1]:
# ============================================================
# DESAFIO — Pesquisador (Linhas 1-Azul, 2-Verde e 3-Vermelha)
# Rode antes, numa célula separada:
#   %pip install -q groq ollama ipywidgets python-dotenv
#
# Como rodar: cole este arquivo inteiro em UMA célula do Colab/VS Code
# (ou quebre nos blocos indicados pelos comentários "# ----------")
# e execute. PROVEDOR = "offline" por padrão, então funciona sem
# nenhuma chave de API. Troque para "groq" ou "ollama" se quiser.
# ============================================================

import os, json, re, unicodedata
from collections import deque
from itertools import product

# ---------- CONFIGURAÇÃO E LLM ----------
PROVEDOR = "offline"        # "groq" (nuvem), "ollama" (local) ou "offline" (sem LLM)
MODELO_GROQ = "openai/gpt-oss-20b"
MODELO_OLLAMA = "llama3.2"

def obter_chave_groq():
    """Busca a chave SEM escrevê-la no código: Colab Secrets → .env → variável de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    return os.environ.get("GROQ_API_KEY")

def chamar_llm(mensagens, modo_json=False):
    """Envia mensagens ao Llama e devolve o texto da resposta."""
    if PROVEDOR == "groq":
        from groq import Groq
        cliente = Groq(api_key=obter_chave_groq())
        extras = {"response_format": {"type": "json_object"}} if modo_json else {}
        resposta = cliente.chat.completions.create(
            model=MODELO_GROQ, messages=mensagens, temperature=0, **extras)
        return resposta.choices[0].message.content
    elif PROVEDOR == "ollama":
        import ollama
        extras = {"format": "json"} if modo_json else {}
        resposta = ollama.chat(model=MODELO_OLLAMA, messages=mensagens,
                                options={"temperature": 0}, **extras)
        return resposta["message"]["content"]
    else:
        raise RuntimeError("Modo offline: nenhum LLM configurado.")


# ---------- DADOS DAS 3 LINHAS (R1) ----------
LINHAS = {
    "Linha 1-Azul": [
        "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
        "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
        "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
        "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
        "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
    ],
    "Linha 2-Verde": [
        "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
        "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",
        "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã", "Tamanduateí",
        "Vila Prudente",
    ],
    "Linha 3-Vermelha": [
        "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",
        "República", "Anhangabaú", "Sé", "Pedro II", "Brás",
        "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",
        "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",
        "Artur Alvim", "Corinthians-Itaquera",
    ],
}

CORES = {"Linha 1-Azul": "#1e88e5", "Linha 2-Verde": "#2e7d32",
         "Linha 3-Vermelha": "#d32f2f"}

# Locais conhecidos: pelo menos 3 por linha (R3)
LOCAIS = {
    # Linha 1-Azul
    "Pinacoteca": "Luz",
    "Catedral da Sé": "Sé",
    "Mosteiro de São Bento": "São Bento",
    # Linha 2-Verde
    "MASP": "Trianon-Masp",
    "Hospital das Clínicas": "Clínicas",
    "Parque do Ibirapuera": "Ana Rosa",
    # Linha 3-Vermelha
    "Theatro Municipal": "Anhangabaú",
    "Neo Química Arena": "Corinthians-Itaquera",
    "Shopping Metrô Tatuapé": "Tatuapé",
}

# Lista simples com as 52 estações, na ordem em que aparecem (sem repetir)
TODAS_ESTACOES = []
for _estacoes in LINHAS.values():
    for _e in _estacoes:
        if _e not in TODAS_ESTACOES:
            TODAS_ESTACOES.append(_e)


# ---------- GRAFO MULTILINHAS (R1) ----------
def construir_grafo_multilinhas(linhas):
    """Retorna (grafo, linhas_do_trecho).
    grafo: {estacao: [vizinhas sem repetição]}  — uma estação que pertence
           a duas linhas (ex.: Sé) é UM ÚNICO nó, o que conecta as linhas.
    linhas_do_trecho: {(a, b): {nomes das linhas}} — guarda (a,b) e (b,a).
    """
    grafo = {}
    linhas_do_trecho = {}
    # 1) garante que toda estação existe no grafo, mesmo antes de ligar arestas
    for estacoes in linhas.values():
        for estacao in estacoes:
            grafo.setdefault(estacao, [])
    # 2) liga cada estação à anterior/próxima da sua linha, sem duplicar
    for nome_linha, estacoes in linhas.items():
        for i in range(len(estacoes) - 1):
            a, b = estacoes[i], estacoes[i + 1]
            if b not in grafo[a]:
                grafo[a].append(b)
            if a not in grafo[b]:
                grafo[b].append(a)
            linhas_do_trecho.setdefault((a, b), set()).add(nome_linha)
            linhas_do_trecho.setdefault((b, a), set()).add(nome_linha)
    return grafo, linhas_do_trecho

GRAFO, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)


# ---------- BFS / DFS (R2) — mesmas funções da aula, reaproveitadas ----------
def reconstruir_caminho(pai, destino):
    caminho = []
    atual = destino
    while atual is not None:
        caminho.append(atual)
        atual = pai[atual]
    return list(reversed(caminho))

def bfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft()
        ordem_visita.append(atual)
        if atual == destino:
            return reconstruir_caminho(pai, destino), ordem_visita
        for vizinho in grafo[atual]:
            if vizinho not in pai and vizinho not in bloqueadas:
                pai[vizinho] = atual
                fila.append(vizinho)
    return None, ordem_visita

def dfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    visitados = set()
    ordem_visita = []
    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino:
            return caminho
        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho])
                if resultado:
                    return resultado
        return None
    return explorar(origem, [origem]), ordem_visita


def contar_baldeacoes(caminho, linhas_do_trecho):
    """Retorna (quantidade, lista de (estacao, linha_nova)).
    Percorre os trechos do caminho mantendo a 'linha atual'; se ela não
    serve mais o próximo trecho, troca de linha e registra a baldeação
    na estação onde a troca acontece. Prefere continuar na mesma linha
    sempre que ela ainda atende o próximo trecho."""
    if not caminho or len(caminho) < 2:
        return 0, []
    baldeacoes = []
    linha_atual = None
    for a, b in zip(caminho, caminho[1:]):
        linhas_possiveis = linhas_do_trecho.get((a, b), set())
        if linha_atual not in linhas_possiveis:
            nova_linha = sorted(linhas_possiveis)[0]
            if linha_atual is not None:
                baldeacoes.append((a, nova_linha))
            linha_atual = nova_linha
    return len(baldeacoes), baldeacoes


# ---------- LÓGICA PROPOSICIONAL (tabela-verdade — R3) ----------
def pode_fazer_baldeacao(integracao, bloqueada, tempo_suficiente):
    """integracao: a estação é ponto de integração entre linhas
       bloqueada: a estação está fechada/bloqueada
       tempo_suficiente: o passageiro tem tempo hábil para a troca
       pode_fazer_baldeacao ≡ integracao ∧ ¬bloqueada ∧ tempo_suficiente"""
    return integracao and (not bloqueada) and tempo_suficiente

def tabela_verdade_baldeacao():
    print(f"{'Integração':<12}{'Bloqueada':<12}{'Tempo OK':<12}{'Pode baldear'}")
    print("-" * 48)
    for integracao, bloqueada, tempo_ok in product([True, False], repeat=3):
        print(f"{str(integracao):<12}{str(bloqueada):<12}{str(tempo_ok):<12}"
              f"{pode_fazer_baldeacao(integracao, bloqueada, tempo_ok)}")


# ---------- LÓGICA DE PRIMEIRA ORDEM: fatos, regras, motor (R3, R6, R7) ----------
def fatos_base():
    """Fatos fixos do mundo: estações, a(s) linha(s) de cada uma, e o que
    fica perto de cada estação."""
    fatos = set()
    for nome_linha, estacoes in LINHAS.items():
        for estacao in estacoes:
            fatos.add(("estacao", estacao))
            fatos.add(("pertence", estacao, nome_linha))
    for local, estacao in LOCAIS.items():
        fatos.add(("proximo_de", local, estacao))
    return fatos

def consultar(fatos, predicado):
    """Devolve os argumentos de todos os fatos de um predicado."""
    return [f[1:] for f in fatos if f[0] == predicado]

# R1 — ∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))
def r_origem(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_esta_em"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("origem", e))
    for (e,) in consultar(fatos, "usuario_esta_na_estacao"):
        novos.add(("origem", e))
    return novos

# R2 — ∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))
def r_destino(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_quer_ir"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("destino", e))
    for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):
        novos.add(("destino", e))
    return novos

# R3 — ∀e (fechada(e) → bloqueada(e))
def r_bloqueio(fatos):
    return {("bloqueada", e) for (e,) in consultar(fatos, "fechada")}

# R4 — ∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))
def r_acessibilidade(fatos):
    if not consultar(fatos, "precisa_acessibilidade"):
        return set()
    return {("inacessivel", e) for (e,) in consultar(fatos, "elevador_em_manutencao")}

# R5 — ∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))
def r_alerta(fatos):
    novos = set()
    inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}
    for papel in ("origem", "destino"):
        for (e,) in consultar(fatos, papel):
            if e in inacessiveis:
                novos.add(("alerta", papel, e))
    return novos

# R6 (OBRIGATÓRIA) — ∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1≠l2 → integracao(e))
def r_integracao(fatos):
    por_estacao = {}
    for estacao, linha in consultar(fatos, "pertence"):
        por_estacao.setdefault(estacao, set()).add(linha)
    return {("integracao", e) for e, linhas in por_estacao.items() if len(linhas) > 1}

# R7 (CRIADA PELO GRUPO) — linha paralisada bloqueia todas as suas estações
# ∀l ∀e (linha_paralisada(l) ∧ pertence(e,l) → bloqueada(e))
def r_linha_paralisada(fatos):
    paralisadas = {l for (l,) in consultar(fatos, "linha_paralisada")}
    if not paralisadas:
        return set()
    return {("bloqueada", e) for e, l in consultar(fatos, "pertence") if l in paralisadas}

REGRAS = [
    ("R1 origem", "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))", r_origem),
    ("R2 destino", "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))", r_destino),
    ("R3 bloqueio", "∀e (fechada(e) → bloqueada(e))", r_bloqueio),
    ("R4 acessibilidade", "∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))", r_acessibilidade),
    ("R5 alerta", "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))", r_alerta),
    ("R6 integracao", "∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1≠l2 → integracao(e))", r_integracao),
    ("R7 linha_paralisada", "∀l ∀e (linha_paralisada(l) ∧ pertence(e,l) → bloqueada(e))", r_linha_paralisada),
]

def encadear_para_frente(fatos, regras, verbose=False):
    """Aplica as regras em rodadas até não surgir nenhum fato novo (ponto fixo)."""
    fatos = set(fatos)
    justificativas = {}
    rodada = 0
    while True:
        rodada += 1
        novos_na_rodada = set()
        for nome, _formula, regra in regras:
            for fato in regra(fatos) - fatos:
                novos_na_rodada.add(fato)
                justificativas[fato] = nome
        if verbose:
            print(f"Rodada {rodada}: {len(novos_na_rodada)} fato(s) novo(s)")
        if not novos_na_rodada:
            return fatos, justificativas
        fatos |= novos_na_rodada


# ---------- PLANEJADOR (lógica + busca + baldeações) ----------
TEMPO_POR_TRECHO = 2       # minutos por trecho (simulado)
TEMPO_POR_BALDEACAO = 5    # minutos extras por troca de linha (simulado)

def planejar(pedido, fechadas=(), manutencao=(), algoritmo="BFS", linhas_paralisadas=()):
    """pedido = {"origem": (tipo, nome), "destino": (tipo, nome), "acessibilidade": bool}
    tipo é "local" ou "estacao"."""
    fatos = fatos_base()
    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]
    fatos.add(("usuario_esta_em", nome_o) if tipo_o == "local" else ("usuario_esta_na_estacao", nome_o))
    fatos.add(("usuario_quer_ir", nome_d) if tipo_d == "local" else ("usuario_quer_ir_estacao", nome_d))
    if pedido.get("acessibilidade"):
        fatos.add(("precisa_acessibilidade",))
    for e in fechadas:
        fatos.add(("fechada", e))
    for e in manutencao:
        fatos.add(("elevador_em_manutencao", e))
    for l in linhas_paralisadas:
        fatos.add(("linha_paralisada", l))

    fatos, justificativas = encadear_para_frente(fatos, REGRAS)

    origem = consultar(fatos, "origem")[0][0]
    destino = consultar(fatos, "destino")[0][0]
    bloqueadas = {e for (e,) in consultar(fatos, "bloqueada")}
    alertas = consultar(fatos, "alerta")
    integracoes = {e for (e,) in consultar(fatos, "integracao")}

    buscar = bfs if algoritmo == "BFS" else dfs
    caminho, visitados = buscar(GRAFO, origem, destino, bloqueadas)
    if caminho:
        n_baldeacoes, pontos_baldeacao = contar_baldeacoes(caminho, LINHAS_DO_TRECHO)
        paradas = len(caminho) - 1
        tempo = paradas * TEMPO_POR_TRECHO + n_baldeacoes * TEMPO_POR_BALDEACAO
    else:
        n_baldeacoes, pontos_baldeacao, paradas, tempo = None, [], None, None

    return {
        "origem": origem, "destino": destino, "algoritmo": algoritmo,
        "caminho": caminho, "visitados": visitados,
        "bloqueadas": sorted(bloqueadas),
        "alertas": [f"{papel}: {e}" for papel, e in alertas],
        "paradas": paradas, "tempo_min": tempo,
        "baldeacoes": n_baldeacoes, "pontos_baldeacao": pontos_baldeacao,
        "integracoes_no_caminho": sorted(integracoes & set(caminho)) if caminho else [],
        "regras_usadas": sorted(set(justificativas.values())),
    }


# ---------- INTÉRPRETE (Llama + guardrails + modo offline) ----------
def normalizar(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def resolver_nome(nome):
    """GUARDRAIL: só aceita nomes que existem de verdade nas 3 linhas. Senão, None."""
    if not nome:
        return None
    alvo = normalizar(nome).strip()
    for estacao in TODAS_ESTACOES:
        if normalizar(estacao) == alvo:
            return ("estacao", estacao)
    for local in LOCAIS:
        if normalizar(local) == alvo:
            return ("local", local)
    return None

PROMPT_INTERPRETE = """Você é o módulo de INTERPRETAÇÃO do Pesquisador SP (Linhas 1, 2 e 3).
Sua única tarefa é transformar o pedido do passageiro em JSON.

Estações válidas: {estacoes}
Locais válidos: {locais}

Responda APENAS com um JSON neste formato:
{{"origem": "<nome exato de estação ou local, ou null>",
  "destino": "<nome exato de estação ou local, ou null>",
  "acessibilidade": <true ou false>}}

Regras:
- Use SOMENTE nomes das listas acima, escritos exatamente como aparecem.
- "acessibilidade" é true se o passageiro mencionar cadeira de rodas,
  mobilidade reduzida, muletas, carrinho de bebê ou precisar de elevador.
- Se não souber algum campo, use null. Nunca invente nomes."""

def interpretar_offline(texto):
    """Plano B sem LLM: procura nomes conhecidos no texto, na ordem em que aparecem."""
    texto_min = texto.lower()
    texto_sem = normalizar(texto)
    candidatos = [(n, "estacao") for n in TODAS_ESTACOES] + [(n, "local") for n in LOCAIS]
    candidatos.sort(key=lambda c: len(c[0]), reverse=True)
    ocupado = [False] * len(texto_min)
    encontrados = []
    for nome, _tipo in candidatos:
        buscas = [(texto_min, nome.lower())]
        if len(nome) > 4 and len(texto_sem) == len(texto_min):
            buscas.append((texto_sem, normalizar(nome)))
        for base, padrao in buscas:
            for m in re.finditer(r"(?<!\w)" + re.escape(padrao) + r"(?!\w)", base):
                if not any(ocupado[m.start():m.end()]):
                    encontrados.append((m.start(), nome))
                    for i in range(m.start(), m.end()):
                        ocupado[i] = True
    encontrados.sort()
    palavras_acess = ["cadeira de rodas", "acessibilidade", "mobilidade",
                       "muleta", "carrinho de bebe", "elevador"]
    return {
        "origem": encontrados[0][1] if len(encontrados) > 0 else None,
        "destino": encontrados[1][1] if len(encontrados) > 1 else None,
        "acessibilidade": any(p in texto_sem for p in palavras_acess),
    }

def interpretar_pedido(texto):
    """Texto livre → pedido validado. Usa o Llama; se falhar, cai no modo offline."""
    if PROVEDOR == "offline":
        bruto = interpretar_offline(texto)
        fonte = "offline"
    else:
        sistema = PROMPT_INTERPRETE.format(
            estacoes=", ".join(TODAS_ESTACOES), locais=", ".join(LOCAIS))
        try:
            resposta = chamar_llm([{"role": "system", "content": sistema},
                                    {"role": "user", "content": texto}], modo_json=True)
            bruto = json.loads(resposta)
            fonte = PROVEDOR
        except Exception as erro:
            print(f"⚠️ LLM indisponível ({erro}). Usando modo offline.")
            bruto = interpretar_offline(texto)
            fonte = "offline"
    origem = resolver_nome(bruto.get("origem"))
    destino = resolver_nome(bruto.get("destino"))
    if origem is None or destino is None:
        return None, f"Não entendi origem/destino (resposta bruta: {bruto})"
    pedido = {"origem": origem, "destino": destino,
              "acessibilidade": bool(bruto.get("acessibilidade"))}
    return pedido, f"Interpretado via {fonte}"


# ---------- NARRADOR ----------
def narrar_offline(r):
    if r["caminho"] is None:
        motivo = f"com as estações bloqueadas: {', '.join(r['bloqueadas'])}" if r["bloqueadas"] else "mesmo sem bloqueios diretos (a rede ficou desconectada nesse trecho)"
        return f"Não existe rota de {r['origem']} até {r['destino']} {motivo}."
    texto = (f"Embarque em {r['origem']} e siga até {r['destino']}: "
             f"{r['paradas']} parada(s), cerca de {r['tempo_min']} minutos.")
    if r["baldeacoes"]:
        pontos = "; ".join(f"em {e}, pegue a {l}" for e, l in r["pontos_baldeacao"])
        texto += f" São necessárias {r['baldeacoes']} baldeação(ões): {pontos}."
    else:
        texto += " Não é necessário trocar de linha."
    if r["alertas"]:
        texto += " Atenção: " + "; ".join(r["alertas"]) + " (elevador em manutenção)."
    return texto

PROMPT_NARRADOR = """Você é o NARRADOR do Pesquisador SP. Explique a rota ao passageiro
em português, em no máximo 5 frases curtas e simpáticas.
Use SOMENTE os dados do JSON. Não invente horários, linhas, estações ou atrações.
Se "caminho" for null, explique que não há rota e cite as estações bloqueadas.
Se "baldeacoes" for maior que zero, cite claramente em qual(is) estação(ões)
o passageiro deve trocar de linha, usando "pontos_baldeacao". Se houver "alertas",
destaque-os."""

def narrar(resultado):
    dados = {k: resultado[k] for k in
             ("origem", "destino", "caminho", "paradas", "tempo_min",
              "bloqueadas", "alertas", "baldeacoes", "pontos_baldeacao")}
    if PROVEDOR == "offline":
        return narrar_offline(resultado)
    try:
        return chamar_llm([{"role": "system", "content": PROMPT_NARRADOR},
                            {"role": "user", "content": json.dumps(dados, ensure_ascii=False)}])
    except Exception as erro:
        return narrar_offline(resultado) + f" (narrador offline: {erro})"


# ---------- VISUALIZAÇÃO: as 3 linhas coloridas ----------
def desenhar_linhas(resultado):
    caminho = set(resultado["caminho"] or [])
    visitados = set(resultado["visitados"])
    bloqueadas = set(resultado["bloqueadas"])
    pontos_baldeacao = {e for e, _ in resultado["pontos_baldeacao"]}
    blocos = []
    for nome_linha, estacoes in LINHAS.items():
        cor_linha = CORES[nome_linha]
        linhas_html = [f"<h4 style='margin:6px 0;color:{cor_linha}'>{nome_linha}</h4>"]
        for estacao in estacoes:
            if estacao in bloqueadas:
                cor, marca = "#757575", "🚫 bloqueada"
            elif estacao in (resultado["origem"], resultado["destino"]) and estacao in caminho:
                cor, marca = "#0d47a1", "⭐ " + ("origem" if estacao == resultado["origem"] else "destino")
            elif estacao in pontos_baldeacao:
                cor, marca = "#f9a825", "🔁 baldeação"
            elif estacao in caminho:
                cor, marca = cor_linha, "rota"
            elif estacao in visitados:
                cor, marca = "#bdbdbd", "visitada pela busca"
            else:
                cor, marca = "#e0e0e0", ""
            linhas_html.append(
                f"<div style='display:flex;align-items:center;gap:8px;font-family:sans-serif;font-size:13px'>"
                f"<span style='display:inline-block;width:14px;height:14px;border-radius:50%;background:{cor}'></span>"
                f"<span style='min-width:190px'>{estacao}</span><span style='color:#666'>{marca}</span></div>")
        blocos.append("<div style='border-left:4px solid " + cor_linha +
                       ";padding-left:8px;min-width:260px'>" + "".join(linhas_html) + "</div>")
    return "<div style='display:flex;flex-wrap:wrap;gap:24px'>" + "".join(blocos) + "</div>"


# ---------- TESTES AUTOMATIZADOS (R6 do desafio) ----------
def rodar_testes():
    # 0. Grafo: 52 estações, sem duplicatas de vizinhos
    assert len(GRAFO) == 52
    assert len(set(GRAFO["Sé"])) == len(GRAFO["Sé"])          # sem repetição
    assert "Ana Rosa" in GRAFO["Paraíso"] and GRAFO["Paraíso"].count("Ana Rosa") == 1

    # 1. Tucuruvi → Corinthians-Itaquera (normal): 22 paradas, 1 baldeação (Sé)
    r = planejar({"origem": ("estacao", "Tucuruvi"), "destino": ("estacao", "Corinthians-Itaquera")})
    assert r["paradas"] == 22 and r["baldeacoes"] == 1

    # 2. Vila Madalena → Jabaquara (normal): 14 paradas, 1 baldeação (Paraíso ou Ana Rosa)
    r = planejar({"origem": ("estacao", "Vila Madalena"), "destino": ("estacao", "Jabaquara")})
    assert r["paradas"] == 14 and r["baldeacoes"] == 1

    # 3. Palmeiras-Barra Funda → Vila Prudente (normal): 16 paradas, 2 baldeações
    r = planejar({"origem": ("estacao", "Palmeiras-Barra Funda"), "destino": ("estacao", "Vila Prudente")})
    assert r["paradas"] == 16 and r["baldeacoes"] == 2

    # 4. Tucuruvi → Brás, Sé fechada: sem rota
    r = planejar({"origem": ("estacao", "Tucuruvi"), "destino": ("estacao", "Brás")}, fechadas={"Sé"})
    assert r["caminho"] is None

    # 5. Vila Madalena → Jabaquara, Paraíso fechada: sem rota (Linha 2 fica cortada)
    r = planejar({"origem": ("estacao", "Vila Madalena"), "destino": ("estacao", "Jabaquara")}, fechadas={"Paraíso"})
    assert r["caminho"] is None

    # 6. Vila Prudente → Jabaquara, Paraíso fechada: 13 paradas, desvio via Ana Rosa
    r = planejar({"origem": ("estacao", "Vila Prudente"), "destino": ("estacao", "Jabaquara")}, fechadas={"Paraíso"})
    assert r["paradas"] == 13 and "Ana Rosa" in r["caminho"]

    # 7. R6 — integração deduzida automaticamente (Sé, Paraíso, Ana Rosa)
    fatos = fatos_base()
    fatos, _ = encadear_para_frente(fatos, REGRAS)
    integracoes = {e for (e,) in consultar(fatos, "integracao")}
    assert integracoes == {"Sé", "Paraíso", "Ana Rosa"}

    # 8. R7 — linha paralisada bloqueia todas as suas estações
    r = planejar({"origem": ("estacao", "Vila Madalena"), "destino": ("estacao", "Vila Prudente")},
                 linhas_paralisadas=["Linha 2-Verde"])
    assert r["caminho"] is None

    print("✅ Todos os 8 testes passaram!")

rodar_testes()


# ---------- EXPLICAÇÃO: por que o caso 5 não tem rota, mas o caso 6 tem? ----------
# Na Linha 2-Verde, a ordem das estações é:
#   ... Brigadeiro, Paraíso, Ana Rosa, Chácara Klabin ... Vila Prudente
# Vila Madalena fica na PONTA OPOSTA da linha em relação a Vila Prudente.
# Para chegar a Ana Rosa (ou a qualquer estação depois dela) partindo de
# Vila Madalena, o trem OBRIGATORIAMENTE passa por Paraíso — não existe
# outro caminho dentro da Linha 2, que é uma linha reta. Por isso, fechar
# Paraíso "corta" a linha ao meio e isola Vila Madalena do restante da rede
# (caso 5: sem rota).
# Já Vila Prudente está do OUTRO lado de Ana Rosa (mais longe de Paraíso).
# Para chegar a Ana Rosa partindo de Vila Prudente, o trem NÃO precisa
# passar por Paraíso. Ana Rosa também é estação de integração com a
# Linha 1-Azul, então o pesquisador consegue seguir por Ana Rosa → Linha 1
# até Jabaquara, contornando a estação fechada (caso 6: 13 paradas, desvio).


# ============================================================
# A PARTIR DAQUI: INTERFACE COM IPYWIDGETS (rode em notebook)
# ============================================================
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

opcoes = ([(f"📍 {local}", ("local", local)) for local in LOCAIS] +
          [(f"🚇 {estacao}", ("estacao", estacao)) for estacao in TODAS_ESTACOES])

txt_pedido = widgets.Textarea(
    placeholder="Ex.: Estou na Catedral da Sé e quero ir ao Neo Química Arena",
    layout=widgets.Layout(width="95%", height="60px"))
btn_interpretar = widgets.Button(description="💬 Interpretar pedido", button_style="info")
dd_origem = widgets.Dropdown(options=opcoes, description="Origem:")
dd_destino = widgets.Dropdown(options=opcoes, value=("estacao", "Vila Prudente"), description="Destino:")
chk_acess = widgets.Checkbox(description="Preciso de acessibilidade")
sel_fechadas = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Fechadas:", rows=6)
sel_manut = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Elevador ⚙️:", rows=6)
sel_paralisadas = widgets.SelectMultiple(options=list(LINHAS.keys()), description="Linha ⛔:", rows=3)
rb_algoritmo = widgets.RadioButtons(options=["BFS", "DFS"], description="Busca:")
btn_buscar = widgets.Button(description="🚇 Buscar rota", button_style="success")
saida = widgets.Output()

def ao_interpretar(_):
    with saida:
        clear_output()
        pedido, msg = interpretar_pedido(txt_pedido.value)
        print(msg)
        if pedido:
            dd_origem.value = pedido["origem"]
            dd_destino.value = pedido["destino"]
            chk_acess.value = pedido["acessibilidade"]
            print("✅ Campos preenchidos. Confira e clique em 'Buscar rota'.")

def ao_buscar(_):
    with saida:
        clear_output()
        pedido = {"origem": dd_origem.value, "destino": dd_destino.value,
                  "acessibilidade": chk_acess.value}
        r = planejar(pedido, sel_fechadas.value, sel_manut.value,
                     rb_algoritmo.value, sel_paralisadas.value)
        display(HTML(f"<h4>{r['algoritmo']}: {r['origem']} → {r['destino']}</h4>"))
        print("🦙", narrar(r))
        if r["caminho"]:
            print(f"🔎 Estações visitadas pela busca: {len(r['visitados'])}")
            print(f"🔁 Baldeações: {r['baldeacoes']}  |  Pontos: {r['pontos_baldeacao']}")
        print(f"📜 Regras disparadas: {', '.join(r['regras_usadas'])}")
        display(HTML(desenhar_linhas(r)))

btn_interpretar.on_click(ao_interpretar)
btn_buscar.on_click(ao_buscar)

painel = widgets.VBox([
    widgets.HTML("<h3>Pesquisa — Linhas 1, 2 e 3</h3>"),
    txt_pedido, btn_interpretar,
    widgets.HBox([dd_origem, dd_destino]),
    widgets.HBox([chk_acess, rb_algoritmo]),
    widgets.HBox([sel_fechadas, sel_manut, sel_paralisadas]),
    btn_buscar, saida,
])

display(painel)

✅ Todos os 8 testes passaram!
